# ERCOT - RTM Price EDA
**Raw source:** `ROOT/data/raw/ercot/RTM_2021_2026Mar/...`  
**Data source:** `ROOT/data/parquet/ercot/dam_lzhb_spp_2021_2025.parquet` 

**Output sources:** `ROOT/data/RTM prices cleaned/..`
- "ercot_rtm_prices_by_settlement_2021_2025.csv"
- "rtm_price_aggregated_2021_2025.csv"

## 1. Imports

In [15]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# Paths
PROJECT_ROOT = Path('../..').resolve()
PARQUET = PROJECT_ROOT / '01_data' / '1.2_raw_api' / 'dam_lzhb_spp_2021_2025.parquet'
OUT_DIR = PROJECT_ROOT / '01_data' / '2_cleaned' / 'rtm_price'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Parquet: {PARQUET}')
print(f'Output : {OUT_DIR}')

Parquet: /Users/zyliazhang/Git/RESEARCH-PJM-ERCOT-Price-Volatility/data/parquet/ercot/dam_lzhb_spp_2021_2025.parquet
Output : /Users/zyliazhang/Git/RESEARCH-PJM-ERCOT-Price-Volatility/data/RTM price cleaned


## 2. Load & inspect with DuckDB

In [16]:
con = duckdb.connect()
# validate time range and column names

column_schema = con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{PARQUET}')
""").df()
print('=== Schema ===')
print(column_schema.to_string(index=False))

=== Schema ===
       column_name column_type null  key default extra
     delivery_date        DATE  YES None    None  None
       hour_ending      BIGINT  YES None    None  None
repeated_hour_flag     BOOLEAN  YES None    None  None
  settlement_point     VARCHAR  YES None    None  None
             price      DOUBLE  YES None    None  None


In [17]:
# Check the time range of the data, reference schema names
time_range = con.execute(f"""
    SELECT 
        MIN(delivery_date) AS min_date
        ,MAX(delivery_date) AS max_date
        ,MIN(hour_ending) AS min_hour
        ,MAX(hour_ending) AS max_hour
    FROM read_parquet('{PARQUET}')
""").fetchone()
print('\n=== Time Range ===')
print(f'{time_range[0]} to {time_range[1]}')
print(f'Hours: {time_range[2]} to {time_range[3]}')


=== Time Range ===
2021-01-01 to 2025-12-27
Hours: 1 to 24


In [18]:
# check settlement locations
settlement_points = con.execute(f"""
    SELECT DISTINCT settlement_point
    FROM read_parquet('{PARQUET}')
""").fetchall()
print('=== Settlement Locations ===')

for loc in settlement_points:
    print(loc[0])
print(f'Number of settlement points: {len(settlement_points)}')

=== Settlement Locations ===
HB_HOUSTON
LZ_AEN
LZ_CPS
HB_NORTH
LZ_RAYBN
HB_HUBAVG
HB_PAN
HB_SOUTH
LZ_LCRA
HB_WEST
LZ_NORTH
HB_BUSAVG
LZ_HOUSTON
LZ_SOUTH
LZ_WEST
Number of settlement points: 15


In [19]:
# Preview raw data
preview = con.execute(f"""
    SELECT *
    FROM read_parquet('{PARQUET}')
    ORDER BY settlement_point, delivery_date, hour_ending
    LIMIT 10
""").df()

print('=== First 10 rows (sorted) ===')
preview

=== First 10 rows (sorted) ===


,delivery_date,hour_ending,repeated_hour_flag,settlement_point,price
0,2021-01-01,1,False,HB_BUSAVG,18.24
1,2021-01-01,2,False,HB_BUSAVG,17.63
2,2021-01-01,3,False,HB_BUSAVG,17.11
3,2021-01-01,4,False,HB_BUSAVG,17.23
4,2021-01-01,5,False,HB_BUSAVG,17.30
5,2021-01-01,6,False,HB_BUSAVG,18.14
6,2021-01-01,7,False,HB_BUSAVG,18.40
7,2021-01-01,8,False,HB_BUSAVG,19.85
8,2021-01-01,9,False,HB_BUSAVG,24.64
9,2021-01-01,10,False,HB_BUSAVG,21.85


In [20]:
# Parse timestamps, drop DST repeat-hour duplicates, sort chronologically
df_prices = con.execute(f"""
    SELECT
        settlement_point
        ,CAST(CONCAT(CAST(delivery_date AS VARCHAR(10)), ' ', CAST(hour_ending AS VARCHAR(2)), ':00') AS DATETIME) AS date_time
        ,price
    FROM read_parquet('{PARQUET}')
    WHERE repeated_hour_flag = 'False'
    ORDER BY settlement_point, date_time
""").df()

print(f'Number of prices found: {len(df_prices):,}')
df_prices

Number of prices found: 649,725


,settlement_point,date_time,price
0,HB_BUSAVG,2021-01-01 01:00:00,18.24
1,HB_BUSAVG,2021-01-01 02:00:00,17.63
2,HB_BUSAVG,2021-01-01 03:00:00,17.11
3,HB_BUSAVG,2021-01-01 04:00:00,17.23
4,HB_BUSAVG,2021-01-01 05:00:00,17.30
...,...,...,...
649720,LZ_WEST,2025-12-27 20:00:00,18.72
649721,LZ_WEST,2025-12-27 21:00:00,16.72
649722,LZ_WEST,2025-12-27 22:00:00,16.11
649723,LZ_WEST,2025-12-27 23:00:00,15.92


In [23]:
# export cleaned data
df_prices.to_csv(OUT_DIR / "ercot_rtm_prices_by_settlement_2021_2025.csv", index=False)

In [22]:
# aggregate system-wide price by date_time
df_system_wide_prices = df_prices.groupby('date_time').agg(
    avg_rtm_price=('price', 'mean'),
    std_rtm_price=('price', 'std')
).reset_index()

df_system_wide_prices.to_csv(OUT_DIR / "rtm_price_aggregated_2021_2025.csv", index=False)
df_system_wide_prices

,date_time,avg_rtm_price,std_rtm_price
0,2021-01-01 01:00:00,18.714000,1.508854
1,2021-01-01 02:00:00,18.107333,1.512517
2,2021-01-01 03:00:00,17.570000,1.537424
3,2021-01-01 04:00:00,17.670667,1.540869
4,2021-01-01 05:00:00,17.792000,1.997417
...,...,...,...
43310,2025-12-27 20:00:00,17.682000,4.066851
43311,2025-12-27 21:00:00,15.002667,4.357721
43312,2025-12-27 22:00:00,13.227333,4.750020
43313,2025-12-27 23:00:00,14.328667,5.222993
